In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ \text{softmax} $$

$$ \mathbb{R}^n \to \mathbb{R}, \quad y_{i}(\mathbf{x}) = \frac{e^{x_i}}{\sum_{k} e^{x_k}} $$

$$ \mathbb{R}^n \to \mathbb{R}^n, \quad y(\mathbf{x}) = (y_1(\mathbf{x}), y_2(\mathbf{x}), \ldots, y_n(\mathbf{x})) $$

$$
\frac{\partial y_i}{\partial x_j} = y_i (\delta_{ij} - y_j) =
\begin{dcases}
    i = j & \frac{\partial y_i}{\partial x_i} = 
    \frac{e^{x_i} \sum_{k} e^{x_k} - e^{x_i} e^{x_i}}{\sum_{k} e^{x_k} \sum_{k} e^{x_k}} =
    y_i (1 - y_i) \\
    \\
    i \neq j & \frac{\partial y_i}{\partial x_j} = 
    \frac{e^{x_j}}{\sum_{k} e^{x_k} \sum_{k} e^{x_k}} = 
    y_i (0 - y_j)\\
\end{dcases}
$$

$$
J_y(\mathbf{x}) =
\text{diag}(y) - y y^\top =
\begin{pmatrix}
    y_1 (1 - y_1) & -y_1 y_2 & \cdots & -y_1 y_n \\
    -y_2 y_1 & y_2 (1 - y_2) & \cdots & -y_2 y_n \\
    \vdots & \vdots & \ddots & \vdots \\
    -y_n y_1 & -y_n y_2 & \cdots & y_n (1 - y_n) \\
\end{pmatrix}
$$

$$ d\mathbf{y} = J_y(\mathbf{x}) \, d\mathbf{x} $$

$$ \text{Frobenius equality} $$

$$
\left \langle \frac{dF}{dx}, d\mathbf{x} \right \rangle =
\left \langle \frac{dF}{dy}, d\mathbf{y} \right \rangle
$$

$$ \\[2em] $$

$$
\left \langle \frac{dF}{dy}, d\mathbf{y} \right \rangle =
$$

$$
\left \langle \frac{dF}{dy}, J_y(x) \, d\mathbf{x} \right \rangle =
$$

$$
\left \langle \frac{dF}{dy}, \Big(\text{diag}(y) - y y^\top \Big) \, d\mathbf{x} \right \rangle =
$$

$$
\left \langle \Big(\text{diag}(y) - y y^\top \Big)^\top \, \frac{dF}{dy}, d\mathbf{x} \right \rangle
$$

$$
\left \langle \Big(\text{diag}(y) - y y^\top \Big) \, \frac{dF}{dy}, d\mathbf{x} \right \rangle
$$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
from approx import approx # type: ignore


def _softmax(x: tr.Tensor, dim: 1) -> tr.Tensor:
    """
        Numerically stable implementation of the `softmax`.
        For simplicity, only the case of `dim=1` is supported.    
    """

    assert dim == 1

    ex = tr.exp(x)
    return ex / tr.sum(ex, dim=-1, keepdim=True)


def softmax(x: tr.Tensor, dim: int = 1) -> tr.Tensor:
    """
        Numerically stable implementation of the `softmax`.
        For simplicity, only the case of `dim=1` is supported.    
    """
    
    assert dim == 1

    x = tr.as_tensor(x)
    return _softmax(x - tr.max(x, dim=-1, keepdim=True).values, dim=1)


class SoftmaxFunction(autograd.Function):
    """Custom autograd function for the `softmax`."""

    @staticmethod
    def forward(ctx, x: tr.Tensor) -> tr.Tensor:
        y = softmax(x, dim=1)
        ctx.save_for_backward(y)
        return y
    
    @staticmethod
    def backward(ctx, dF_dy: tr.Tensor) -> tuple[tr.Tensor,]:
        (y, ) = ctx.saved_tensors

        dot = (dF_dy * y).sum(dim=1, keepdim=True)
        dF_dx = y * (dF_dy - dot)

        return (dF_dx, )
    

class Softmax(nn.Module):
    """Custom module for the `softmax`."""
    
    def __init__(self) -> None:
        super().__init__()

    def forward(self, x: tr.Tensor) -> tr.Tensor:
        return SoftmaxFunction.apply(x)

In [ ]:
# Unit tests

def test_basic() -> None:
    assert softmax(1000, dim=1) == approx(1.0)
    assert softmax(100, dim=1) == approx(1.0)
    assert softmax(10, dim=1) == approx(1.0)
    assert softmax(1, dim=1) == approx(1.0)


def test_gradcheck(samples) -> None:
    def func(x: tr.Tensor) -> tr.Tensor:
        return SoftmaxFunction.apply(x)

    x = tr.randn(samples, 3, dtype=tr.float64, requires_grad=True)
    assert autograd.gradcheck(func, (x, ), eps=0.001, atol=0.001)


def test_compare(samples) -> None:
    x = tr.randn(samples, 3, dtype=tr.float32, requires_grad=True)

    x1 = x.clone().detach().requires_grad_(True)
    y1 = Softmax()(x1)
    F1 = y1.sum()
    F1.backward()

    x2 = x.clone().detach().requires_grad_(True)
    y2 = tr.softmax(x2, dim=-1)
    F2 = y2.sum()
    F2.backward()

    assert y1 == approx(y2)
    assert x1.grad == approx(x2.grad)


if __name__ == "__main__":
    test_basic()

    test_gradcheck(1)
    test_gradcheck(10)
    test_gradcheck(100)

    test_compare(1)
    test_compare(10)
    test_compare(100)